## Section 1: Project header and load.

# Module 4 Project Brief: The FinFlow Churn Investigation

**Analyst:** Meron Welderufael
**Date:** 11/18/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** This project analyzes FinFlow's enriched customer dataset to investigate customer churn, compare churn patterns across plan tiers and signup cohorts, and identify trends for the dashboard. The notebook includes data loading, cleaning, exploratory analysis, and interactive visualizations using Python and Plotly.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-enriched-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

print(f"Customer Data: {df.shape}")

Customer Data: (200000, 21)


### Data Loading and Initial Setup

Import the required libraries, configure pandas display settings, and load the dataset while parsing date columns into datetime format. Confirm that the dataset contains 200,000 rows and 21 columns.

## Section 2: Six-method inspection.

In [2]:
# Shape: how many rows and columns?
print("Shape:",df.shape)

Shape: (200000, 21)


In [3]:
# Data types per column
print(df.dtypes)

customer_id                                str
first_name                                 str
last_name                                  str
email                                      str
city                                       str
state                                      str
country                                    str
signup_date                     datetime64[us]
plan_tier                                  str
monthly_fee                            float64
last_login_date                 datetime64[us]
total_sessions                           int64
avg_session_duration_minutes           float64
support_tickets_opened                   int64
active                                    bool
churn_date                      datetime64[us]
referral_source                            str
marketing_name                             str
target_segment                             str
monthly_fee_listed                     float64
launch_date                                str
dtype: object

In [4]:
# First five rows
df.head()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date,referral_source,marketing_name,target_segment,monthly_fee_listed,launch_date
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2026-03-11,Basic,9.99,2026-10-14,91,11.80,4,True,NaT,Organic Search,Starter Plan,Individual,9.99,2019-03-15
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-12-06,Basic,9.99,2026-10-05,328,3.30,1,True,NaT,Social Media,Starter Plan,Individual,9.99,2019-03-15
2,FF-00003,Jeffery,Wagner,jeffery.wagner3@example.com,New Jamesside,MT,US,2026-04-04,Basic,9.99,2026-10-13,23,12.10,1,True,NaT,Social Media,Starter Plan,Individual,9.99,2019-03-15
3,FF-00004,Jeffrey,Lawrence,jeffrey.lawrence4@example.com,Port Lindachester,MA,US,2025-10-25,Basic,9.99,2026-06-27,6,2.90,5,False,2026-10-08,Word of Mouth,Starter Plan,Individual,9.99,2019-03-15
4,FF-00005,Joshua,Lewis,joshua.lewis5@example.com,Curtisfurt,CT,US,2026-02-25,Pro,29.99,2026-10-04,83,14.70,3,True,NaT,Organic Search,Professional,SMB,29.99,2020-08-01


In [5]:
# Last five rows
df.tail()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date,referral_source,marketing_name,target_segment,monthly_fee_listed,launch_date
199995,FF-199996,Mitchell,Johnson,mitchell.johnson199996@example.com,North Ronald,KY,US,2025-12-13,Pro,29.99,2026-10-02,54,5.50,1,True,NaT,Paid Search,Professional,SMB,29.99,2020-08-01
199996,FF-199997,Crystal,Thompson,crystal.thompson199997@example.com,Carolynborough,MO,US,2025-08-12,Pro,29.99,2026-09-22,159,4.70,3,True,NaT,Social Media,Professional,SMB,29.99,2020-08-01
199997,FF-199998,Erik,Wiley,erik.wiley199998@example.com,Jeffreyhaven,KS,US,2025-09-07,Basic,9.99,2026-10-09,400,11.60,3,True,NaT,Affiliate,Starter Plan,Individual,9.99,2019-03-15
199998,FF-199999,Jonathan,Arias,jonathan.arias199999@example.com,Kellyfurt,NM,US,2025-11-27,Basic,9.99,2026-10-08,111,10.40,4,True,NaT,Organic Search,Starter Plan,Individual,9.99,2019-03-15
199999,FF-200000,Jeffrey,Johnson,jeffrey.johnson200000@example.com,Brownview,WY,US,2026-01-03,Basic,9.99,2026-09-30,94,3.80,4,True,NaT,Organic Search,Starter Plan,Individual,9.99,2019-03-15


In [6]:
# Full info dump
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 21 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   customer_id                   200000 non-null  str           
 1   first_name                    200000 non-null  str           
 2   last_name                     200000 non-null  str           
 3   email                         200000 non-null  str           
 4   city                          200000 non-null  str           
 5   state                         200000 non-null  str           
 6   country                       200000 non-null  str           
 7   signup_date                   200000 non-null  datetime64[us]
 8   plan_tier                     200000 non-null  str           
 9   monthly_fee                   200000 non-null  float64       
 10  last_login_date               200000 non-null  datetime64[us]
 11  total_sessions          

In [7]:
# Numeric summary statistics
df.describe()

,signup_date,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,churn_date,monthly_fee_listed
count,200000,200000.00,200000,200000.00,200000.00,200000.00,39998,200000.00
mean,2025-12-21 11:06:28.079999,34.98,2026-09-18 01:18:40.032000,173.27,7.89,2.41,2026-08-29 11:47:58.523926,34.98
min,2024-10-22 00:00:00,9.99,2026-04-24 00:00:00,1.00,1.00,0.00,2026-04-24 00:00:00,9.99
25%,2025-09-03 00:00:00,9.99,2026-09-22 00:00:00,48.00,4.60,1.00,2026-08-05 00:00:00,9.99
50%,2026-01-12 00:00:00,9.99,2026-10-02 00:00:00,163.00,7.50,2.00,2026-09-08 00:00:00,9.99
75%,2026-04-22 00:00:00,29.99,2026-10-12 00:00:00,282.00,11.20,4.00,2026-09-30 00:00:00,29.99
max,2026-10-19 00:00:00,199.00,2026-10-21 00:00:00,400.00,15.00,8.00,2026-10-21 00:00:00,199.00
std,NaN,55.56,NaN,123.19,3.86,1.89,NaN,55.56


### Dataset Inspection Summary

The six inspection methods were used to examine the dataset's structure, data types, sample records, and numerical distributions. The dataset contains **200,000 rows and 21 columns**, with approximately 55.1 MB of memory usage. The columns include 13 string, 3 datetime, 3 float, 2 integer, and 1 Boolean data types. The `signup_date`, `last_login_date`, and `churn_date` columns were successfully parsed into datetime format. Most columns have complete records, while `churn_date` contains 39,998 non-null values, which is expected because customers who have not churned do not have a recorded churn date. Overall, the dataset structure looks consistent and suitable for further customer churn analysis.

## Section 3: Per-tier churn breakdown.

In [8]:
# Active rate by plan tier
active_rate_by_tier = (
    df.groupby('plan_tier')['active']
    .mean()
    .mul(100)
    .reset_index(name='active_rate_pct')
)

# Churn rate by plan tier
churn_rate_by_tier = (
    (1 - df.groupby('plan_tier')['active'].mean()).mul(100)
        .reset_index(name='churn_rate_pct'))

print('Active rate by plan tier:')
display(active_rate_by_tier)

print('Churn rate by plan tier:')
display(churn_rate_by_tier) 

Active rate by plan tier:


,plan_tier,active_rate_pct
0,Basic,74.92
1,Enterprise,95.04
2,Pro,85.14


Churn rate by plan tier:


,plan_tier,churn_rate_pct
0,Basic,25.08
1,Enterprise,4.96
2,Pro,14.86


### Churn Rates by Plan Tier

The analysis shows clear differences in customer retention across plan tiers. Enterprise has the highest active rate at **95.04%** and the lowest churn rate at **4.96%**, indicating strong customer retention. Pro follows with an active rate of **85.14%** and a churn rate of **14.86%**. Basic has the lowest active rate at **74.92%** and the highest churn rate at **25.08%**, making it the tier with the greatest customer retention challenge. These results suggest that Basic customers are more likely to churn, while Enterprise customers demonstrate stronger retention.

## Section 4: Signup-cohort breakdown.

In [9]:
#signup month using the Period dtype
df['signup_cohort'] = df['signup_date'].dt.to_period('M')

def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

cohort_churn = df.groupby('signup_cohort').agg(
    churn_rate_pct = ('active', churn_rate_pct)  
).round(2)
print(cohort_churn.head(10))
print('---rows omitted---')
print(cohort_churn.tail(5))

               churn_rate_pct
signup_cohort                
2024-10                 11.67
2024-11                 19.77
2024-12                 19.00
2025-01                 18.83
2025-02                 19.73
2025-03                 21.05
2025-04                 20.01
2025-05                 20.49
2025-06                 20.20
2025-07                 19.77
---rows omitted---
               churn_rate_pct
signup_cohort                
2026-06                 20.21
2026-07                 19.62
2026-08                 20.49
2026-09                 21.05
2026-10                 19.15



### Signup Cohort Churn Analysis

The monthly signup cohort analysis shows that the observed churn rates remain relatively consistent, generally ranging between approximately 19% and 21% across most cohorts. However, these results should be interpreted with caution because cohorts have different lengths of time to experience churn. Older cohorts have had more time to churn, while newer cohorts may have only been customers for a few days or weeks. Therefore, comparing their current churn rates directly could be misleading. This analysis does not establish whether customer retention improves or declines over time. A more meaningful comparison would measure churn at the same customer age, such as the six-month churn rate after signup. This is an important follow-up analysis to flag for Diego rather than a conclusion from the current deliverable.


## Section 5: Cross-cut, cohort by tier

In [10]:
#Cross-cut, cohort by tier
cohort_by_tier = (
    df.groupby(['signup_cohort', 'marketing_name'])
    .agg(
    churn_rate_pct = ('active', churn_rate_pct))
    .round(2)
    .reset_index()
)

print('Table shape:', cohort_by_tier.shape)
print(cohort_by_tier.head(5))
print()
print(cohort_by_tier.tail(5))

Table shape: (75, 3)
  signup_cohort   marketing_name  churn_rate_pct
0       2024-10  Enterprise Plus            0.00
1       2024-10     Professional           10.00
2       2024-10     Starter Plan           15.62
3       2024-11  Enterprise Plus            4.65
4       2024-11     Professional           12.35

   signup_cohort   marketing_name  churn_rate_pct
70       2026-09     Professional           16.81
71       2026-09     Starter Plan           25.57
72       2026-10  Enterprise Plus            1.61
73       2026-10     Professional           18.08
74       2026-10     Starter Plan           22.83


### Cohort Churn by Marketing Plan

The cross-cut analysis compares churn rates across signup cohorts and marketing plan categories, resulting in 75 cohort-plan combinations. The results show differences in churn across the three plans: Starter Plan generally has higher churn rates, Professional falls in the middle, and Enterprise Plus tends to have the lowest churn. For example, in the October 2026 cohort, Starter Plan has a churn rate of 22.83%, Professional 18.08%, and Enterprise Plus 1.61%. However, newer cohorts have had less time to experience churn, so these differences should be interpreted cautiously. This breakdown helps identify potential retention patterns across plan categories, but comparing churn at equal customer ages would provide a more reliable assessment.


## Section 6: Summary block


## Key Findings and Summary

1. **Basic has the highest churn rate:** Basic customers have a churn rate of **25.08%**, compared with **14.86%** for Pro and **4.96%** for Enterprise. This makes Basic the plan tier with the greatest customer retention challenge.

2. **Enterprise demonstrates the strongest retention:** Enterprise has an active rate of **95.04%**, meaning only **4.96%** of its customers have churned. This is substantially better than Basic (74.92% active) and Pro (85.14% active).

3. **Overall cohort churn is relatively consistent:** Monthly signup cohorts generally show churn rates around **19%–21%**. However, this comparison does not account for differences in customer age, so it cannot establish whether newer or older cohorts retain customers better.

4. **Starter Plan consistently shows higher churn:** In the October 2026 cohort, Starter Plan recorded **22.83%** churn, compared with **18.08%** for Professional and **1.61%** for Enterprise Plus. This suggests differences in retention across marketing plan categories, although cohort age should be considered.

5. **Equal-age cohort analysis is a recommended follow-up:** Current cohort churn rates compare customers with different lengths of exposure to churn. Measuring churn at the same customer age, such as six months after signup, would provide a fairer comparison and help determine whether retention is improving or declining.
